# TP1 — Extraire une réclamation client en JSON

**Contexte** : vous travaillez sur l'assistant support client de **Lagune Banque** (données fictives).
Les clients écrivent des réclamations en texte libre, parfois en français ivoirien, avec des abréviations SMS.
Le but est de transformer chaque message en un **objet JSON** exploitable par la suite du pipeline.

**Champs à extraire** : `categorie`, `reference_transaction`, `montant_fcfa`, `demande`, `sentiment`.

**Durée totale indicative** : 1 h 15. Les cellules à compléter sont marquées `# TODO` avec un indice.
La dernière cellule contrôle automatiquement votre travail.

## Étape 0 — Configuration

**Objectif** : renseigner `MODELE` (et la clé `LLM_API_KEY`) dans le fichier `starter/.env`, puis vérifier la connexion.  
**Durée indicative** : 5 min  
**Résultat attendu** : la cellule de test affiche `Connexion OK`.

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import litellm
import pandas as pd
from dotenv import load_dotenv

# Le notebook peut être lancé depuis starter/ ou depuis son propre dossier : on cherche le .env.
for dossier in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "../../starter"]:
    if (dossier / ".env").exists():
        load_dotenv(dossier / ".env")
        break

In [ ]:
MODELE = os.getenv("MODELE")
MODELE_2 = os.getenv("MODELE_2") or None  # optionnel
API_KEY = os.getenv("LLM_API_KEY")
litellm.drop_params = True  # ignore les paramètres que le modèle ne supporte pas

if not (MODELE and API_KEY):
    print("⚠ TODO : renseignez LLM_API_KEY et MODELE dans starter/.env, puis relancez cette cellule.")
else:
    try:
        r = litellm.completion(model=MODELE, api_key=API_KEY, max_tokens=20,
                               messages=[{"role": "user", "content": "Réponds : OK"}])
        print("Connexion OK :", MODELE)
    except Exception as e:
        print(f"ÉCHEC de connexion ({type(e).__name__}). Vérifiez MODELE et LLM_API_KEY.")

## Étape 1 — Les données

**Objectif** : lire les 10 réclamations et leur référence (le « gold », écrit à la main avant tout essai de modèle).  
**Durée indicative** : 5 min  
**Résultat attendu** : un tableau de 10 lignes : message et valeurs attendues.

In [ ]:
CANDIDATS = [Path("m1/tp1_extraction_json/data"), Path("data"), Path("../../starter/m1/tp1_extraction_json/data")]
DATA = next(p for p in CANDIDATS if (p / "reclamations.jsonl").exists())

with open(DATA / "reclamations.jsonl", encoding="utf-8") as f:
    MESSAGES = [json.loads(ligne) for ligne in f if ligne.strip()]
with open(DATA / "gold.json", encoding="utf-8") as f:
    GOLD = json.load(f)
GOLD_PAR_ID = {g["id"]: g for g in GOLD}
CHAMPS = ["categorie", "reference_transaction", "montant_fcfa", "demande", "sentiment"]

In [ ]:
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(GOLD).set_index("id").join(pd.DataFrame(MESSAGES).set_index("id"))[["message", *CHAMPS]]

## Étape 2 — Le prompt

**Objectif** : écrire `PROMPT` : rôle, champs et valeurs autorisées, exemple de sortie, règle du `null`, mot « UNIQUEMENT ».  
**Durée indicative** : 15 min  
**Résultat attendu** : une chaîne de caractères qui décrit précisément le JSON attendu.

In [ ]:
# Champs attendus (voir le gold ci-dessus) :
#   categorie              : probleme_virement | probleme_compte | carte_bancaire | facturation | autre
#   reference_transaction  : "VIR-XXXXX" ou null
#   montant_fcfa           : entier ou null
#   demande                : regularisation_virement | remboursement | assistance | information
#   sentiment              : satisfait | neutre | mecontent
PROMPT = None
# TODO : écrire le prompt système. Indice : donnez un rôle, listez chaque champ avec ses valeurs
# autorisées, ajoutez un exemple de sortie, dites quoi faire si l'information est absente, et
# demandez de répondre UNIQUEMENT avec le JSON.
if PROMPT is None:
    raise NotImplementedError("À compléter")

## Étape 3 — Appel et parsing

**Objectif** : appeler le modèle avec `litellm.completion`, puis transformer sa réponse en dictionnaire Python.  
**Durée indicative** : 15 min  
**Résultat attendu** : `extraire(...)` renvoie `(dict ou None, latence, tokens_in, tokens_out)` pour un message.

In [ ]:
def parser_json(brut):
    """Extrait l'objet JSON d'une réponse, même entourée de texte ou de balises Markdown."""
    if not brut:
        return None
    # TODO : retirer les balises ```json ... ```, ne garder que le texte entre la première « { »
    # et la dernière « } », puis json.loads. Indice : str.find, str.rfind, try/except json.JSONDecodeError.
    raise NotImplementedError("À compléter")

In [ ]:
BRUT = {}  # dernière réponse brute, pour analyser les échecs


def extraire(texte, modele, **options):
    debut = time.perf_counter()
    reponse = None
    # TODO : appeler litellm.completion. Indice : model=modele, api_key=API_KEY, temperature=0,
    # messages = [message système (PROMPT), message utilisateur (texte)], et **options.
    if reponse is None:
        raise NotImplementedError("À compléter")
    latence = time.perf_counter() - debut
    BRUT["dernier"] = reponse.choices[0].message.content
    usage = reponse.usage
    return parser_json(BRUT["dernier"]), latence, usage.prompt_tokens, usage.completion_tokens

In [ ]:
sortie, latence, tin, tout = extraire(MESSAGES[0]["message"], MODELE)
print(sortie)
print(f"{latence:.1f} s, {tin} tokens en entrée, {tout} tokens en sortie")

## Étape 4 — Mesures

**Objectif** : compléter `score(sortie, attendu)` (nombre de champs justes sur 5), puis évaluer le modèle sur les 10 messages.  
**Durée indicative** : 10 min  
**Résultat attendu** : pour chaque message : un score sur 5, une latence et un coût.

In [ ]:
def score(sortie, attendu):
    """Nombre de champs justes sur 5 (0 si la sortie n'est pas un dictionnaire)."""
    # TODO : comparer chaque champ de CHAMPS entre sortie et attendu. Indice : sortie.get(champ) == attendu[champ]
    raise NotImplementedError("À compléter")

In [ ]:
# Prix en USD par million de tokens : (entrée, sortie). Utilisé seulement si litellm ne connaît pas le modèle.
PRIX = {}  # exemple : PRIX = {"mon-modele": (0.50, 1.50)}


def cout(modele, tin, tout):
    """Coût d'un appel en USD, ou None si le prix du modèle est inconnu."""
    try:
        c_in, c_out = litellm.cost_per_token(model=modele, prompt_tokens=tin, completion_tokens=tout)
        return c_in + c_out
    except Exception:
        if modele in PRIX:
            return (tin * PRIX[modele][0] + tout * PRIX[modele][1]) / 1_000_000
        return None

In [ ]:
def evaluer(modele, etiquette=None, **options):
    resultats = []
    for rec in MESSAGES:
        attendu = GOLD_PAR_ID[rec["id"]]
        sortie, latence, tin, tout = extraire(rec["message"], modele, **options)
        resultats.append({"id": rec["id"], "message": rec["message"], "sortie": sortie, "attendu": attendu,
                          "score": score(sortie, attendu), "latence": latence,
                          "cout": cout(modele, tin, tout), "brut": BRUT["dernier"]})
    return {"modele": etiquette or modele, "resultats": resultats}


EVALS = [evaluer(MODELE)]
print("Évaluation terminée :", len(EVALS[0]["resultats"]), "messages")

## Étape 5 — Le tableau de synthèse

**Objectif** : résumer l'évaluation : JSON valides, champs justes, latence moyenne, coût pour 1 000 réclamations.  
**Durée indicative** : 5 min  
**Résultat attendu** : un tableau pandas d'une ligne par modèle.

In [ ]:
def resume(evals):
    lignes = []
    for ev in evals:
        res = ev["resultats"]
        couts = [r["cout"] for r in res]
        lignes.append({
            "Modèle": ev["modele"],
            "JSON valide /10": sum(r["sortie"] is not None for r in res),
            "Champs justes /50": sum(r["score"] for r in res),
            "Latence moy. (s)": round(sum(r["latence"] for r in res) / len(res), 2),
            "Coût / 1 000 (USD)": None if None in couts else round(sum(couts) / len(couts) * 1000, 3),
        })
    return pd.DataFrame(lignes).set_index("Modèle")

In [ ]:
resume(EVALS)

## Étape 6 — Analyser les échecs

**Objectif** : comprendre pourquoi certains champs sont faux : JSON invalide, montant faux, écart entre modèles.  
**Durée indicative** : 10 min  
**Résultat attendu** : une explication écrite pour chaque type d'échec observé.

In [ ]:
def afficher_echecs(ev):
    for r in ev["resultats"]:
        if r["score"] == 5:
            continue
        print(f"--- id {r['id']} ({r['score']}/5) : {r['message'][:80]}")
        if r["sortie"] is None:
            print("  JSON invalide, réponse brute :", (r["brut"] or "")[:200])
            continue
        for champ in CHAMPS:
            if r["sortie"].get(champ) != r["attendu"][champ]:
                print(f"  {champ} : attendu {r['attendu'][champ]!r}, obtenu {r['sortie'].get(champ)!r}")


afficher_echecs(EVALS[0])

### Vos observations

*TODO : pour chaque échec affiché ci-dessus, notez la cause (JSON invalide ? montant mal converti ?
référence inventée ? consigne du client suivie ? valeur hors liste ?) et ce que vous changeriez dans le prompt.*

## Étape 7 — Option : un second modèle

**Objectif** : relancer l'évaluation avec `MODELE_2` (à renseigner dans `.env`) et comparer.  
**Durée indicative** : 5 min  
**Résultat attendu** : une ligne de plus dans le tableau.

In [ ]:
if MODELE_2:
    EVALS.append(evaluer(MODELE_2))
else:
    print("MODELE_2 non renseigné dans .env : étape ignorée.")
resume(EVALS)

## Étape 8 — Bonus : validation Pydantic

**Objectif** : définir le schéma `Reclamation` pour rejeter automatiquement les sorties non conformes.  
**Durée indicative** : 10 min  
**Résultat attendu** : une colonne « Taux de rejet » dans le tableau.

In [ ]:
from typing import Literal, Optional

from pydantic import BaseModel, ConfigDict, ValidationError


class Reclamation(BaseModel):
    model_config = ConfigDict(extra="forbid")
    # TODO : déclarer les 5 champs. Indice : Literal["a", "b"] pour les listes de valeurs,
    # Optional[str] et Optional[int] pour les champs qui peuvent valoir null (sans valeur par défaut).

In [ ]:
def rejete(sortie):
    """Une sortie est rejetée si elle n'est pas du JSON ou ne respecte pas le schéma (types stricts)."""
    if sortie is None:
        return True
    try:
        Reclamation.model_validate_json(json.dumps(sortie), strict=True)
        return False
    except ValidationError:
        return True


tableau = resume(EVALS)
tableau["Taux de rejet"] = [sum(rejete(r["sortie"]) for r in ev["resultats"]) / len(ev["resultats"]) for ev in EVALS]
tableau

## Étape 9 — Conclusion

**Objectif** : décider si le modèle convient pour ce cas d'usage.  
**Durée indicative** : 5 min  
**Résultat attendu** : une phrase qui cite le coût pour 1 000 réclamations et le taux de rejet.

### Votre conclusion

*TODO : en une phrase, ce modèle convient-il ? Citez le coût pour 1 000 réclamations et le taux de rejet.*

## Contrôle automatique

Cette dernière cellule affiche **ÉCHEC** tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    res = EVALS[0]["resultats"]
    assert len(res) == 10 and all(isinstance(r["sortie"], dict) for r in res), \
        "toutes les sorties doivent être des dictionnaires"
    total = sum(r["score"] for r in res)
    print(f"Champs justes : {total}/50 (objectif indicatif : au moins 40/50)")
    print("OK" if total >= 40 else "Objectif indicatif non atteint : améliorez le prompt.")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")